# Assignment 1 2AMM10 2025-2026

## Group: [DL group]
### Member 1: [Hannes Janmaat 1548999]
### Member 2: [Benjamin Softic 1573608]
### Member 3: [Fill in your name]

## Task 1

Dataset and visualization

In [ ]:
import os
import re
from pathlib import Path
from torch.utils.data import Dataset
from PIL import Image
import kagglehub
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

class AppleDataset(Dataset):
    def __init__(self, transform=None, subset="train",class_subset = "main"):
        assert subset in ["train","test"]
        assert class_subset in ["main","new","all"]
        path = kagglehub.dataset_download("moltean/fruits")
        base = Path(path) / "fruits-360_original-size" / "fruits-360-original-size"
        if subset == "train":
            self.path = base / "Training"
        elif subset == "test":
            self.path = base / "Validation"
        self.transform = transform
        all_folders = sorted(os.listdir(self.path))
        self.item_folders = sorted(x for x in all_folders if x.lower().startswith("apple"))
        generator=np.random.default_rng(6)
        generator.shuffle(self.item_folders)
        if class_subset == "main":
            self.item_folders = self.item_folders[:20]
        elif class_subset == "new":
            self.item_folders = self.item_folders[20:]
        self.targets = []
        self.image_paths = []
        for i, folder in enumerate(self.item_folders):
            for img_file in sorted(os.listdir(self.path / folder)):
                if img_file.startswith("r0"):
                    if class_subset=="new":
                        self.targets.append(i+20)
                    else:
                        self.targets.append(i)
                    self.image_paths.append(self.path / folder / img_file)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        image = Image.open(self.image_paths[i]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.targets[i]

dataset = AppleDataset()

item_dd = widgets.Dropdown(options=dataset.item_folders, description="Variety:")
frame_slider = widgets.IntSlider(value=0, min=0, max=0, description="Frame:")
output = widgets.Output()


def get_frames(folder):
    return sorted(f for f in os.listdir(dataset.path / folder)
                  if f.startswith("r0_") and f.endswith(".jpg"))

def update_slider(*_):
    frames = get_frames(item_dd.value)
    frame_slider.max = max(0, len(frames) - 1)
    frame_slider.value = min(frame_slider.value, frame_slider.max)
    show_image()

def show_image(*_):
    frames = get_frames(item_dd.value)
    if not frames or frame_slider.value >= len(frames):
        return
    with output:
        clear_output(wait=True)
        img = Image.open(dataset.path / item_dd.value / frames[frame_slider.value])
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(img)
        ax.set_title(f"{item_dd.value} | frame {frame_slider.value}")
        ax.axis("off")
        plt.tight_layout()
        plt.show()

item_dd.observe(update_slider, names="value")
frame_slider.observe(show_image, names="value")

update_slider()
display(widgets.VBox([item_dd, frame_slider, output]))

Using Colab cache for faster access to the 'fruits' dataset.


In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.RandomAffine(degrees=8, translate=(0.04, 0.04), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
eval_transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
train_data = AppleDataset(subset="train", transform=train_transform)
test_data = AppleDataset(subset="test", transform=eval_transform)
support_new_data = AppleDataset(subset="train", transform=eval_transform, class_subset="new")
test_new_data = AppleDataset(subset="test", transform=eval_transform, class_subset="new")

Using Colab cache for faster access to the 'fruits' dataset.
Using Colab cache for faster access to the 'fruits' dataset.
Using Colab cache for faster access to the 'fruits' dataset.
Using Colab cache for faster access to the 'fruits' dataset.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EmbeddingNet(nn.Module):
    def __init__(self):
        super(EmbeddingNet, self).__init__()

        self.front_layer = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=2, stride=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=5, stride=5),
            nn.Conv2d(64, 128, kernel_size=2, stride=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=5, stride=1),
        )

        self.last_layer = nn.Linear(128, 32)

    def forward(self, x):
        x = self.front_layer(x)
        x = torch.flatten(x, start_dim=1)
        x = self.last_layer(x)
        x = F.normalize(x, p=2, dim=1)
        return x

    def get_embedding(self, x):
        return self.forward(x)

In [ ]:
from torch.utils.data.sampler import BatchSampler
import numpy as np
class BalancedBatchSampler(BatchSampler):
    def __init__(self, labels, n_classes, n_samples):
        self.labels = labels
        self.labels_set = list(set(self.labels))
        self.label_to_indices = {label: np.where(  np.array(self.labels) == label)[0]
                                 for label in self.labels_set}
        for l in self.labels_set:
            np.random.shuffle(self.label_to_indices[l])
        self.used_label_indices_count = {label: 0 for label in self.labels_set}
        self.count = 0
        self.n_classes = n_classes
        self.n_samples = n_samples
        self.n_dataset = len(self.labels)
        self.batch_size = self.n_samples * self.n_classes

    def __iter__(self):
        self.count = 0
        while self.count + self.batch_size < self.n_dataset:
            classes = np.random.choice(self.labels_set, self.n_classes, replace=False)
            indices = []
            for class_ in classes:
                indices.extend(self.label_to_indices[class_][
                               self.used_label_indices_count[class_]:self.used_label_indices_count[
                                                                         class_] + self.n_samples])
                self.used_label_indices_count[class_] += self.n_samples
                if self.used_label_indices_count[class_] + self.n_samples > len(self.label_to_indices[class_]):
                    np.random.shuffle(self.label_to_indices[class_])
                    self.used_label_indices_count[class_] = 0
            yield indices
            self.count += self.n_classes * self.n_samples

    def __len__(self):
        return self.n_dataset // self.batch_size

In [ ]:
train_batch_sampler = BalancedBatchSampler(train_data.targets, n_classes=10, n_samples=6)

triplets_train_loader = torch.utils.data.DataLoader(train_data, batch_sampler=train_batch_sampler)

In [ ]:
from itertools import combinations

def pdist(vectors):
    distance_matrix = -2 * vectors.mm(torch.t(vectors)) + vectors.pow(2).sum(dim=1).view(1, -1) + vectors.pow(2).sum(
        dim=1).view(-1, 1)
    return distance_matrix

class Informative_Negative_TripletSelector():

    def __init__(self, margin):
        super(Informative_Negative_TripletSelector, self).__init__()

        self.margin = margin

   # Our goal is to mining informative triplets.
    def informative_negative(self, loss_values):

        informative_negative = np.where(loss_values > 0)[0]
        return np.random.choice(informative_negative) if len(informative_negative) > 0 else None


    def get_triplets(self, embeddings, labels):

        if torch.cuda.is_available()==False:
            embeddings = embeddings.cpu()
        distance_matrix = pdist(embeddings)
        distance_matrix = distance_matrix.cpu()

        labels = labels.cpu().data.numpy()
        triplets = []

        for label in set(labels):
            label_mask = (labels == label)
            label_indices = np.where(label_mask)[0]
            if len(label_indices) < 2:
                continue
            negative_indices = np.where(np.logical_not(label_mask))[0]
            anchor_positives = list(combinations(label_indices, 2))  # All anchor-positive pairs
            anchor_positives = np.array(anchor_positives)


            ap_distances = distance_matrix[anchor_positives[:, 0], anchor_positives[:, 1]]
            for anchor_positive, ap_distance in zip(anchor_positives, ap_distances):
                loss_values = ap_distance - distance_matrix[torch.LongTensor(np.array([anchor_positive[0]])), torch.LongTensor(negative_indices)] + self.margin
                loss_values = loss_values.data.cpu().numpy()

                informative_negative = self.informative_negative(loss_values)
                if informative_negative is not None:
                    informative_negative = negative_indices[informative_negative]
                    triplets.append([anchor_positive[0], anchor_positive[1], informative_negative])

        if len(triplets) == 0:
            triplets.append([anchor_positive[0], anchor_positive[1], negative_indices[0]])

        triplets = np.array(triplets)

        return torch.LongTensor(triplets)

In [ ]:
class TripletLoss(nn.Module):
    def __init__(self, margin, triplet_selector):
        super(TripletLoss, self).__init__()
        self.margin = margin
        self.triplet_selector = triplet_selector

    def forward(self, embeddings, target):

        triplets = self.triplet_selector.get_triplets(embeddings, target)
        # print("triplets: ", triplets[0].shape)

        if embeddings.is_cuda:
            triplets = triplets.cuda()


        anchor_idx= triplets[:, 0]
        positive_idx= triplets[:, 1]
        negative_idx= triplets[:, 2]

        anchors = embeddings[anchor_idx]
        positives = embeddings[positive_idx]
        negatives = embeddings[negative_idx]

        distance_vectors_to_positive = anchors - positives
        distance_vectors_to_neagtive = anchors - negatives
        norm_squared_of_distance_vectors_to_positive = (distance_vectors_to_positive ** 2).sum(dim=1)
        norm_squared_of_distance_vectors_to_negative = (distance_vectors_to_neagtive ** 2).sum(dim=1)

        margin_vector = torch.tensor([self.margin for i in range(anchors.shape[0])],
    device=device)
        delta_vector = margin_vector + norm_squared_of_distance_vectors_to_positive - norm_squared_of_distance_vectors_to_negative

        zeros = torch.zeros((anchors.shape[0])).to(device)
        zeros_delta = torch.cat([zeros.unsqueeze(1), delta_vector.unsqueeze(1)], dim = 1)
        losses = torch.max(zeros_delta, dim=1).values

        return losses.mean()

In [ ]:
import numpy as np
from tqdm import tqdm


class Trainer():
    def __init__(self,
                 model: torch.nn.Module,
                 device: torch.device,
                 criterion: torch.nn.Module,
                 optimizer: torch.optim.Optimizer,
                 training_DataLoader: torch.utils.data.Dataset,
                 epochs: int
                 ):

        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.training_DataLoader = training_DataLoader
        self.device = device
        self.epochs = epochs

    def run_trainer(self):
        for epoch in tqdm(range(self.epochs)):
            self.model.train()
            train_losses=[]
            for batch in self.training_DataLoader:
                x,y=batch
                input, target = x.to(self.device), y.to(self.device)
                self.optimizer.zero_grad()
                out = self.model(input)
                loss = self.criterion(out, target)
                loss_value = loss.item()
                train_losses.append(loss_value)
                loss.backward()
                self.optimizer.step()

            print(f'EPOCH: {epoch+1:0>{len(str(self.epochs))}}/{self.epochs}', end=' ')
            print(f'LOSS: {np.mean(train_losses):.4f}',end=' ')

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device=torch.device('cpu')

mined_embedding_net = EmbeddingNet()
mined_model = mined_embedding_net.to(device)
margin=.5
criterion = TripletLoss(margin,  Informative_Negative_TripletSelector(margin))
optimizer = torch.optim.Adam(mined_model.parameters(), lr=1e-3)
trainer = Trainer(model=mined_model,
                  device=device,
                  criterion=criterion,
                  optimizer=optimizer,
                  training_DataLoader=triplets_train_loader,
                  epochs=20,
                  )
trainer.run_trainer()

  5%|▌         | 1/20 [00:18<05:42, 18.02s/it]

EPOCH: 01/20 LOSS: 0.3012 

 10%|█         | 2/20 [00:35<05:23, 17.98s/it]

EPOCH: 02/20 LOSS: 0.2216 

 15%|█▌        | 3/20 [00:54<05:10, 18.28s/it]

...

EPOCH: 18/20 LOSS: 0.0949 

 95%|█████████▌| 19/20 [05:51<00:18, 18.34s/it]

EPOCH: 19/20 LOSS: 0.1003 

100%|██████████| 20/20 [06:10<00:00, 18.53s/it]

EPOCH: 20/20 LOSS: 0.0775 

In [ ]:
def dist(centroids, embedding):
    result = []

    for label, centroid in centroids.items():
        distance = ((centroid - embedding) ** 2).sum()
        result.append((label, distance))

    result.sort(key=lambda x: x[1])
    return result

def compute_centroids(dataset):
    mined_model.eval()
    labels = np.array(dataset.targets)

    clusters = {}
    for label in np.unique(labels):
        clusters[label] = []
    for ix, label in enumerate(labels):
        img, _ = dataset[ix]
        img = img.unsqueeze(0).to(device)
        with torch.no_grad():
            embedding = mined_model(img).squeeze(0).cpu()
        clusters[label].append(embedding)

    centroids = {}
    for label, cluster in clusters.items():
        cluster = torch.stack(cluster)
        centroid = cluster.mean(dim=0)
        centroid = F.normalize(centroid.unsqueeze(0), p=2, dim=1).squeeze(0)
        centroids[label] = centroid
    return centroids

def nearest_centroid_label(centroids, embedding):
    distances = dist(centroids, embedding)
    return distances[0][0]

def evaluate_nearest_centroid(support_dataset, test_dataset):
    mined_model.eval()
    centroids = compute_centroids(support_dataset)
    correctly_classified = 0
    for x, y in test_dataset:
        input = x.unsqueeze(0).to(device)
        with torch.no_grad():
            embedding = mined_model(input).squeeze(0).cpu()
        prediction = nearest_centroid_label(centroids, embedding)
        if prediction == y:
            correctly_classified += 1
    accuracy = correctly_classified / len(test_dataset)
    return accuracy

seen_accuracy = evaluate_nearest_centroid(support_dataset=train_data, test_dataset=test_data)
unseen_accuracy = evaluate_nearest_centroid(support_dataset=support_new_data, test_dataset=test_new_data)

print("Seen-item test accuracy: ", seen_accuracy)
print("Unseen-item test accuracy: ", unseen_accuracy)

Seen-item test accuracy:  0.9479843953185956
Unseen-item test accuracy:  0.8693982074263764


In [ ]:
torch.save(mined_model.state_dict(), "model_weights.pth")
print(os.getcwd())

## Task 2

In [ ]:
class GardenDataset(Dataset):
    def __init__(self, transform=None, class_level="item", subset="train", family_subset="main", item_subset="main"):
        assert class_level in ["item","family","both"]
        assert subset in ["train","test"]
        assert family_subset in ["main","new","all"]
        assert item_subset in ["main","new","all"]
        path = kagglehub.dataset_download("moltean/fruits")
        base = Path(path) / "fruits-360_original-size" / "fruits-360-original-size"
        if subset == "train":
            self.path = base / "Training"
        elif subset == "test":
            self.path = base / "Validation"
        self.transform = transform
        self.class_level = class_level

        canonical_items = sorted(
            d for d in os.listdir(base / "Training")
            if (base / "Training" / d).is_dir() and re.fullmatch(r'\S+ \d+', d)
        )
        item_to_family = {it: it.rsplit(' ', 1)[0] for it in canonical_items}
        canonical_families = sorted(set(item_to_family.values()))

        self.item_to_idx = {c: i for i, c in enumerate(canonical_items)}
        self.family_to_idx = {c: i for i, c in enumerate(canonical_families)}

        train_fam_to_items = defaultdict(list)
        for it in canonical_items:
            train_fam_to_items[item_to_family[it]].append(it)
        for fam in train_fam_to_items:
            train_fam_to_items[fam].sort(key=lambda x: int(x.rsplit(' ', 1)[1]))

        new_families = {fam for fam, its in train_fam_to_items.items() if len(its) == 1}
        new_items = set()
        for fam, its in train_fam_to_items.items():
            if len(its) >= 3:
                new_items.add(its[0])

        present = {
            d for d in os.listdir(self.path)
            if (self.path / d).is_dir() and re.fullmatch(r'\S+ \d+', d)
        }
        all_items = [it for it in canonical_items if it in present]

        if family_subset == "main":
            all_items = [it for it in all_items if item_to_family[it] not in new_families]
        elif family_subset == "new":
            all_items = [it for it in all_items if item_to_family[it] in new_families]

        if item_subset == "main":
            all_items = [it for it in all_items if it not in new_items]
        elif item_subset == "new":
            all_items = [it for it in all_items if it in new_items]

        self.items = all_items
        self.item_to_family = {it: item_to_family[it] for it in self.items}
        self.families = sorted(set(self.item_to_family.values()))
        self.new_families = new_families
        self.new_items = new_items

        # Build samples using canonical (global) indices
        self.image_paths = []
        self.targets_item = []
        self.targets_family = []
        for item in self.items:
            item_dir = self.path / item
            item_label = self.item_to_idx[item]
            family_label = self.family_to_idx[item_to_family[item]]
            for img_file in sorted(os.listdir(item_dir)):
                if img_file.endswith('.jpg'):
                    self.image_paths.append(item_dir / img_file)
                    self.targets_item.append(item_label)
                    self.targets_family.append(family_label)

        if class_level == "item":
            self.classes = self.items
            self.class_to_idx = self.item_to_idx
            self.targets = self.targets_item
        elif class_level == "family":
            self.classes = self.families
            self.class_to_idx = self.family_to_idx
            self.targets = self.targets_family

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        if self.class_level == "both":
             return image, self.targets_family[idx], self.targets_item[idx]
        return image, self.targets[idx]

    def get_items_for_family(self, family):
        return sorted(item for item, fam in self.item_to_family.items() if fam == family)

dataset = GardenDataset()

family_dd = widgets.Dropdown(options=dataset.families, description="Family:")
item_dd = widgets.Dropdown(options=dataset.get_items_for_family(dataset.families[0]), description="Item:")
frame_slider = widgets.IntSlider(value=0, min=0, max=0, description="Frame:")
output = widgets.Output()


def get_frames(item):
    return sorted(f for f in os.listdir(dataset.path / item) if f.endswith(".jpg"))

def update_items(*_):
    items = dataset.get_items_for_family(family_dd.value)
    item_dd.options = items
    item_dd.value = items[0]

def update_slider(*_):
    frames = get_frames(item_dd.value)
    frame_slider.max = max(0, len(frames) - 1)
    frame_slider.value = min(frame_slider.value, frame_slider.max)
    show_image()

def show_image(*_):
    frames = get_frames(item_dd.value)
    if not frames or frame_slider.value >= len(frames):
        return
    with output:
        clear_output(wait=True)
        img = Image.open(dataset.path / item_dd.value / frames[frame_slider.value])
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(img)
        ax.set_title(f"{family_dd.value} | {item_dd.value} | frame {frame_slider.value}")
        ax.axis("off")
        plt.tight_layout()
        plt.show()

family_dd.observe(update_items, names="value")
item_dd.observe(update_slider, names="value")
frame_slider.observe(show_image, names="value")

update_slider()
display(widgets.VBox([family_dd, item_dd, frame_slider, output]))

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.RandomAffine(degrees=8, translate=(0.04, 0.04), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
eval_transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_data = GardenDataset(subset="train", transform=eval_transform)

# scenario 1
test_data = GardenDataset(subset="test", transform=eval_transform)

# scenario 2
train_data_family = GardenDataset(subset="train", transform=eval_transform, class_level="family")
test_data_family = GardenDataset(subset="test", transform=eval_transform, class_level="family")

# scenario 3
support_all_data = GardenDataset(subset="train", transform=eval_transform, item_subset="all")
test_new_data = GardenDataset(subset="test", transform=eval_transform, item_subset="new")

# scenario 4
support_all_data_family = GardenDataset(subset="train", transform=eval_transform, family_subset="all",class_level="family")
test_new_data_family = GardenDataset(subset="test", transform=eval_transform, family_subset="new",class_level="family")

# your code here

In [ ]:
class EmbeddingNet(nn.Module):
    def __init__(self):
        super(EmbeddingNet, self).__init__()

        self.front_layer = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=2, stride=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=5, stride=5),
            nn.Conv2d(64, 128, kernel_size=2, stride=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=5, stride=1),
        )

        self.last_layer = nn.Linear(128, 128)

    def forward(self, x):
        x = self.front_layer(x)
        x = torch.flatten(x, start_dim=1)
        x = self.last_layer(x)
        x = F.normalize(x, p=2, dim=1)
        return x

    def get_embedding(self, x):
        return self.forward(x)

In [ ]:
train_batch_sampler = BalancedBatchSampler(train_data.targets, n_classes=20, n_samples=6)

triplets_train_loader = torch.utils.data.DataLoader(train_data, batch_sampler=train_batch_sampler)

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device=torch.device('cpu')

mined_embedding_net = EmbeddingNet()
mined_model = mined_embedding_net.to(device)
margin=.5
criterion = TripletLoss(margin,  Informative_Negative_TripletSelector(margin))
optimizer = torch.optim.Adam(mined_model.parameters(), lr=1e-3)
trainer = Trainer(model=mined_model,
                  device=device,
                  criterion=criterion,
                  optimizer=optimizer,
                  training_DataLoader=triplets_train_loader,
                  epochs=20,
                  )
trainer.run_trainer()

In [ ]:
acc_s1 = evaluate_nearest_centroid(support_dataset=train_data, test_dataset=test_data)
print("Scenario 1 test accuracy: ", acc_s1)

In [ ]:
acc_s2 = evaluate_nearest_centroid(support_dataset=train_data_family, test_dataset=test_data_family)
print("Scenario 2 test accuracy: ", acc_s2)

In [ ]:
acc_s3 = evaluate_nearest_centroid(support_dataset=support_all_data, test_dataset=test_new_data)
print("Scenario 3 test accuracy: ", acc_s3)

In [ ]:
acc_s4 = evaluate_nearest_centroid(support_dataset=support_all_data_family, test_dataset=test_new_data_family)
print("Scenario 4 test accuracy: ", acc_s4)

## Task 3

In [ ]:
train_data_both = GardenDataset(class_level="both",transform=transform,subset="train",family_subset="main",item_subset="main")

test_data_both = GardenDataset(class_level="both",transform=transform,subset="test",family_subset="main",item_subset="main")

# your code here

## Task 4

In [ ]:
class BlackoutPixels:
    """Transform that randomly sets x% of pixels to black (0).

    Args:
        fraction: Fraction of pixels to black out (0.0 to 1.0).
    """
    def __init__(self, fraction=0.1):
        self.fraction = fraction

    def __call__(self, img):
        # img shape: (C, H, W)
        _, h, w = img.shape
        num_pixels = h * w
        num_black = int(num_pixels * self.fraction)

        # Random pixel indices to black out
        indices = torch.randperm(num_pixels)[:num_black]
        rows = indices // w
        cols = indices % w

        img = img.clone()
        img[:, rows, cols] = 0.0
        return img

def get_anomaly_dataset(fraction):
    transform = transforms.Compose([
        BlackoutPixels(fraction=fraction),
        ... # your transforms here
    ])
    return GardenDataset(subset="test", transform=transform, family_subset="main", item_subset="main")

# your code here
